# Deliverable 3 — Sistema de agentes en Google Colab (T4)

Mismo modelo en todos los agentes: **Qwen3-4B-Instruct-2507 + LoRA `checkpoint-300`** (el de D2).
Cuatro agentes con rol y prompt propios que **deciden** qué pasa después:

| agente | decide | herramientas | prompt |
|---|---|---|---|
| A1 Triage | continuar / rechazar | `detectar_expresion_temporal` | `a1_triage_v2.txt` |
| A2 Extractor | pasa / se reintenta si su JSON no es válido | — | `a2_extractor_v1.txt` |
| A3 Temporal | qué herramientas llamar, cuándo terminar | `resolver_fecha`, `normalizar_hora`, `calcular_duracion`, `extraer_recordatorio`, `extraer_zona_horaria`, `extraer_recurrencia`, `expandir_participantes` | `a3_agente_v1.txt` |
| A4 Verificador | aprobar / corregir / devolver a A3 | `validar_contrato` | `a4_agente_v1.txt` |

**Brazos** (mismos inputs, mismo criterio `evaluar.py`): `E0` prompt directo (modelo base) · `E1` solución D2 ·
`E4` sistema completo · `E4T1` sin herramientas · `E4T2` sin few-shot · `E4T3` sin rúbrica/CoT · `E4T4` sin A4 (sin bucle) · `E4T5` sin A1.

**Inputs (160):** A = 93 del diagnóstico (`frases.json` + `benchmark_ft.json`) · B = 41 de validación por clase de falla
(`benchmark_heldout.json`) · **C = 26 independientes** (`benchmark_independiente.json`, escritas sin conocer el sistema).

**Qué se reutiliza:** `d3_previos.zip` trae E1 sobre A y B (ya medido) y el caché del modelo (A2 idéntico). Solo se corre
E1 sobre C. E0 se corre con el modelo base (sin LoRA) y luego se descarga de la GPU.

**Tiempo:** ~1,5–2 h en T4. Reanudable: si una celda se corta, vuelve a correrla y salta lo ya hecho. Con `USAR_DRIVE = True` (celda 5) los resultados quedan en tu Google Drive (`d3_intento3_resultados/`) y sobreviven a una caída de la sesión.
Baja `d3_resultados_intento3.zip` al final (celda 11).

In [ ]:
# 1) Dependencias (T4) — mismas versiones con las que se entrenó el LoRA
!nvidia-smi -L
%pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0"
%pip uninstall -y -q torchao
import torch
assert torch.cuda.is_available(), "Sin GPU: cambia el runtime a T4 y vuelve a correr"
print("GPU OK:", torch.cuda.get_device_name(0))


In [ ]:
# 2) Subir d3_proyecto.zip (codigo + prompts + benchmarks) y dejarlo listo
import os, sys, zipfile
from google.colab import files

REPO = "/content/repo"
os.makedirs(REPO, exist_ok=True)
if not os.path.isdir(os.path.join(REPO, "deliverable3")):
    print("Sube d3_proyecto.zip (carpeta subir_a_colab):")
    subidos = files.upload()
    nombre = next(f for f in subidos if f.endswith(".zip"))
    with zipfile.ZipFile(nombre) as z:
        z.extractall(REPO)

os.chdir(REPO)
sys.path.insert(0, REPO)
from deliverable3 import benchmark_agentes as ba   # deja deliverable2/ en sys.path
assert "E4" in ba.BRAZOS, "este zip no es el del intento3"
for b in ("frases", "benchmark_ft", "benchmark_heldout", "benchmark_independiente"):
    assert os.path.exists(os.path.join(REPO, b + ".json")), f"falta {b}.json"
print("OK: deliverable3 listo en", REPO)
print("brazos:", ", ".join(ba.BRAZOS))

In [ ]:
# 3) Verificar el codigo en Colab (tests D3 + 28 tests D2, sin GPU, ~2 s)
import subprocess, sys, re
r1 = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "deliverable3/tests", "-t", "."],
                    cwd=REPO, capture_output=True, text=True)
r2 = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                    cwd=os.path.join(REPO, "deliverable2"), capture_output=True, text=True)
print(r1.stdout[-300:], r1.stderr[-300:])
print(r2.stdout[-300:], r2.stderr[-300:])
assert r1.returncode == 0 and r2.returncode == 0, "tests en rojo: no seguir"
print("tests OK", [int(x) for x in re.findall(r"Ran (\d+) tests", r1.stderr + r2.stderr)])

In [ ]:
# 4) Adapter LoRA checkpoint-300 (el mismo de D2)
ORIGEN_ADAPTER = "subir archivos"  # @param ["subir archivos", "drive"]
RUTA_DRIVE = "/content/drive/MyDrive/qwen3-4b-semantic-lora/checkpoint-300"  # @param {type:"string"}
import os, shutil
from google.colab import files

NECESARIOS = ["adapter_config.json", "adapter_model.safetensors"]

if ORIGEN_ADAPTER == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER = RUTA_DRIVE
else:
    ADAPTER = "/content/adapter"
    os.makedirs(ADAPTER, exist_ok=True)
    faltan = [f for f in NECESARIOS if not os.path.isfile(os.path.join(ADAPTER, f))]
    if faltan:
        print("Sube:", ", ".join(faltan), "(de checkpoint-300; el .safetensors pesa ~130 MB)")
        for nombre in files.upload():
            clave = "adapter_config.json" if "adapter_config" in nombre else "adapter_model.safetensors"
            shutil.move(nombre, os.path.join(ADAPTER, clave))
assert os.path.isfile(os.path.join(ADAPTER, "adapter_config.json")), "falta adapter_config.json"
assert os.path.isfile(os.path.join(ADAPTER, "adapter_model.safetensors")), "falta adapter_model.safetensors"
print("adapter OK:", ADAPTER)


In [ ]:
# 5) Resultados previos reutilizados: E1 sobre A y B (intentos 1 y 2) y el cache del modelo (A2 identico)
import zipfile
from pathlib import Path

D3 = Path(REPO) / "deliverable3"
SALIDA = D3 / "resultados"
TRAZAS = SALIDA / "traces"

# Persistencia: con Drive montado, resultados/ vive en tu Drive y sobrevive a una caida de la sesion de Colab
USAR_DRIVE = True  # @param {type:"boolean"}
if USAR_DRIVE and not SALIDA.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        destino = Path("/content/drive/MyDrive/d3_intento3_resultados")
        destino.mkdir(parents=True, exist_ok=True)
        SALIDA.symlink_to(destino)
        print("resultados/ se guarda en", destino)
    except Exception as e:
        print("sin Drive (se usa el disco de Colab):", e)
SALIDA.mkdir(parents=True, exist_ok=True)
TRAZAS.mkdir(parents=True, exist_ok=True)

print("Sube d3_previos.zip (carpeta subir_a_colab):")
subidos = files.upload()
nombre = next(f for f in subidos if f.endswith(".zip"))
with zipfile.ZipFile(nombre) as z:
    PREVIOS = set(z.namelist())     # no se vuelven a empaquetar al final
    z.extractall(D3)
print(sorted(PREVIOS))

In [ ]:
# 6) E0: prompt directo con el modelo BASE (sin LoRA) sobre los 160 inputs; luego se libera la GPU
import gc, json, time, torch
from argparse import Namespace
from pipeline.modelo import GeneradorHF

MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"
BENCHES = ["frases.json", "benchmark_ft.json", "benchmark_heldout.json", "benchmark_independiente.json"]

def args_de(brazo, bench, traza=None):
    return Namespace(bench=bench, salida=str(SALIDA), motor="hf", modelo=MODELO_ID, adapter=ADAPTER,
                     traza=traza, dir_trazas=str(TRAZAS), fuerza=False, resumen=False)

def ya_corrido(brazo, bench):
    return (SALIDA / ("resultados_%s_%s.json" % (brazo, Path(bench).stem))).exists()

pendientes = [b for b in BENCHES if not ya_corrido("E0", b)]
vram_e0 = None
if pendientes:
    t0 = time.perf_counter()
    motor_base = GeneradorHF(MODELO_ID, adapter=None, dtype="float16")
    print(f"modelo base cargado en {time.perf_counter()-t0:.0f} s")
    torch.cuda.reset_peak_memory_stats()
    for bench in pendientes:
        ba.correr_brazo("E0", args_de("E0", bench), motor_base)
    vram_e0 = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    print("E0 listo | VRAM max:", vram_e0, "GB")
    del motor_base
    gc.collect(); torch.cuda.empty_cache()
else:
    print("E0 ya estaba corrido: se salta")

In [ ]:
# 7) Cargar Qwen3-4B + LoRA UNA vez (~8 GB fp16) y envolverlo en el cache (A2 reutiliza lo del intento 2)
from deliverable3.motor_cache import MotorConCache

t0 = time.perf_counter()
motor_base = GeneradorHF(MODELO_ID, adapter=ADAPTER, dtype="float16")
print(f"cargado en {motor_base.modelo.device} en {time.perf_counter()-t0:.0f} s")
torch.cuda.reset_peak_memory_stats()
motor = MotorConCache(motor_base, SALIDA / "cache_llm.json")
print("entradas en cache:", len(motor.cache))

In [ ]:
# 8) Smoke con modelo real: E4 sobre 2 frases, mostrando los mensajes entre agentes
from deliverable3.grafo import ejecutar
from deliverable3.agentes import BRAZOS

def mostrar(traza, ancho=150):
    for t in traza.turnos:
        if t["tipo"] == "mensaje":
            print(f'  {t["de"]:>32s} -> {t["a"]:<32s} {json.dumps(t["contenido"], ensure_ascii=False)[:ancho]}')
        elif t["tipo"] == "agente":
            print(f'  [{t["agente"]}] {t["salida"][:ancho].replace(chr(10), " ")}')
        elif t["tipo"] == "evento":
            print(f'  => {t["estado"]}: {t["evento"]}')

for frase in ("Reunión el 10/09 a las 10 pm", "Reunión de pauta el viernes 18 a las 10, avísame 15 min antes"):
    print("=" * 100); print(frase)
    evento, traza = ejecutar(frase, cfg=BRAZOS["E4"], motor=motor)
    mostrar(traza)
print("\nsmoke OK (revisa que A3 haya llamado herramientas y que A4 haya decidido)")

In [ ]:
# 8b) PRE-VUELO (~3 min): 12 frases de A por E4. Si algo anda mal con el bucle de A3, se ve aqui y no tras 1,5 h de corrida
import benchmark_ft as bft
fr_pre, meta_pre = bft.cargar(str(ba.RUTA_BENCH / "frases.json"))
n = rechazos = con_fecha = respaldo = guardia = terminar_vacio = 0
pasos, decisiones_a4 = [], {}
for f in fr_pre[:12]:
    ev, tr = ejecutar(f["texto"], cfg=BRAZOS["E4"], motor=motor, fecha_ref=meta_pre["fecha_referencia"])
    n += 1
    ag = [x for x in tr.turnos if x["tipo"] == "agente"]
    rechazos += any(x["agente"] == "A1_Triage" and x.get("accion") == "rechazar" for x in ag)
    fin = [x for x in ag if x["agente"] == "A3_Temporal" and "herramientas" in x][-1]
    con_fecha += "resolver_fecha" in fin["herramientas"]
    respaldo += bool(fin.get("respaldo")); guardia += bool(fin.get("guardia_terminar_vacio"))
    pasos.append(sum(1 for x in ag if x["agente"] == "A3_Temporal" and x.get("accion") in ("llamar", "terminar", "invalida")))
    for x in ag:
        if x["agente"] == "A4_Verificador":
            decisiones_a4[x["accion"]] = decisiones_a4.get(x["accion"], 0) + 1
print(f"A1 rechazos: {rechazos}/{n} | A3 llamo resolver_fecha: {con_fecha}/{n} | respaldo: {respaldo}/{n} | guardia terminar-vacio: {guardia}/{n}")
print(f"A3 pasos promedio: {sum(pasos)/n:.1f} | decisiones de A4: {decisiones_a4}")
ok = rechazos == 0 and con_fecha >= 10 and respaldo <= 3
print("\nPRE-VUELO", "OK: sigue con la demo y la corrida" if ok else
      "CON ALERTA: no sigas; copia esta salida y la de la celda 8 (smoke) para revisarlas antes de gastar la corrida completa")

## 7) DEMO — material del video

Tres inputs reales, uno de cada conjunto. Se ve la secuencia de mensajes entre agentes y herramientas, el bucle de A4 cuando
ocurre, el resultado final y, al lado, lo que da la solución D2 (E1) sobre la misma frase (E0 está en la tabla de la celda 10).
Edita `DEMO` para cambiar las frases. Las trazas JSONL quedan en `deliverable3/resultados/traces/video/`.

In [ ]:
# DEMO — E4 con los mensajes entre agentes visibles; E0 y E1 al lado sobre la misma frase
DEMO = [
    "puedes agendar la reunión de sincronización el 10 a las 14? avísame 15 min antes",       # A: fallaba en D2/E2
    "Entrevista con el candidato a ingeniero el 8 de octubre a las 10:00",                   # B: validación por clase
    "Schedule a call with Priya and the design team on Zoom, October 20 at 2 pm, 45 minutes, remind me 10 minutes before",  # C: independiente
]
TRAZA_VIDEO = SALIDA / "traces" / "video"
TRAZA_VIDEO.mkdir(parents=True, exist_ok=True)
CAMPOS = ["fecha", "hora_inicio", "duracion_minutos", "participantes", "ubicacion", "recordatorio_minutos", "zona_horaria"]
for i, frase in enumerate(DEMO, 1):
    print("=" * 100); print(f"[{i}] {frase}\n")
    e1, _ = ejecutar(frase, cfg=BRAZOS["E1"], motor=motor)
    print("  E1 (solucion D2):", {c: e1[c] for c in CAMPOS})
    e4, traza = ejecutar(frase, cfg=BRAZOS["E4"], motor=motor)
    print("\n  Mensajes entre agentes (E4):")
    mostrar(traza)
    print("\n  E4 (agentes):    ", {c: e4[c] for c in CAMPOS})
    print("  traza ->", traza.guardar(TRAZA_VIDEO / f"demo_{i}_E4.jsonl"))

## 9) Corrida: E4 y las cinco ablaciones sobre los 160 inputs (+ E1 sobre C)

Reanudable: si `resultados_{brazo}_{bench}.json` ya existe, se salta. El caché hace que las ablaciones solo paguen las llamadas que cambian.

In [ ]:
# 9) Correr brazos. Se guardan TODAS las trazas (mensajes entre agentes) en resultados/traces/trazas_{brazo}_{bench}.jsonl
BRAZOS_CORRER = ["E4", "E4T4", "E4T5", "E4T2", "E4T3", "E4T1", "E1"]

t_sesion = time.perf_counter()
for brazo in BRAZOS_CORRER:
    for bench in BENCHES:
        if ya_corrido(brazo, bench):
            print(f"skip {brazo}/{bench} (ya existe)")
            continue
        t0 = time.perf_counter()
        ba.correr_brazo(brazo, args_de(brazo, bench), motor)
        motor.guardar()
        print(f"  {brazo}/{bench} en {(time.perf_counter()-t0)/60:.1f} min | modelo: {motor.llamadas} llamadas, cache: {motor.aciertos} aciertos")
print(f"\nsesion total: {(time.perf_counter()-t_sesion)/3600:.2f} h")

## 10) Resultados

Frases 100 % correctas por conjunto y agregado (n = 160), McNemar exacto de cada brazo contra E1 y de E4 contra cada ablación.
El conjunto A es optimista (el diagnóstico se hizo ahí); **C es la evidencia independiente**. Guarda `comparacion_d3_v4.json`.

In [ ]:
# 10) Tabla + McNemar + comportamiento de los agentes (leido de las trazas)
import benchmark_ft as bft
import evaluar as ev

ARMS = ["E0", "E1", "E4", "E4T1", "E4T2", "E4T3", "E4T4", "E4T5"]
GRUPOS = {"A (93)": ["frases.json", "benchmark_ft.json"], "B (41)": ["benchmark_heldout.json"],
          "C (26, independiente)": ["benchmark_independiente.json"], "TOTAL (160)": BENCHES}

def cargar_res(brazo, bench):
    p = SALIDA / ("resultados_%s_%s.json" % (brazo, Path(bench).stem))
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

def evaluados(brazo, bench):
    r = cargar_res(brazo, bench)
    if r is None:
        return None
    frases, _ = bft.cargar(str(ba.RUTA_BENCH / bench))
    return bft._evaluar(frases, r)

def correctos(brazo, benches):
    e = [evaluados(brazo, b) for b in benches]
    return None if any(x is None for x in e) else [x["es_correcto_total"] for z in e for x in z]

def mcnemar(a, b):
    a_gana = sum(1 for x, y in zip(a, b) if x and not y)
    b_gana = sum(1 for x, y in zip(a, b) if not x and y)
    return a_gana, b_gana, bft._mcnemar(a_gana, b_gana)

tabla = []
for grupo, benches in GRUPOS.items():
    print("\n" + "=" * 90); print(grupo); print("=" * 90)
    base_e1 = correctos("E1", benches)
    for brazo in ARMS:
        c = correctos(brazo, benches)
        if c is None:
            continue
        fila = {"grupo": grupo, "brazo": brazo, "correctos": sum(c), "total": len(c)}
        txt = ""
        if base_e1 is not None and brazo != "E1":
            a, b, p = mcnemar(base_e1, c)
            fila["vs_E1"] = {"E1_gana": a, brazo + "_gana": b, "p": round(p, 4)}
            txt += f"  vs E1: {a}-{b} p={p:.3f}"
        e4 = correctos("E4", benches)
        if e4 is not None and brazo.startswith("E4T"):
            a, b, p = mcnemar(e4, c)
            fila["E4_vs_ablacion"] = {"E4_gana": a, brazo + "_gana": b, "p": round(p, 4)}
            txt += f"  | E4 vs {brazo}: {a}-{b} p={p:.3f}"
        tabla.append(fila)
        print(f"{brazo:5s} {sum(c):3d}/{len(c):<3d}{txt}")

# comportamiento de los agentes en E4 (de las trazas)
comp = {"A1_rechazos": 0, "A2_reintentos": 0, "A3_respaldo": 0, "A3_pasos_promedio": 0.0, "A4_aprobar": 0, "A4_corregir": 0,
        "A4_devolver_a_A3": 0, "A4_correcciones_rechazadas": 0, "inputs": 0}
pasos = []
for bench in BENCHES:
    p = TRAZAS / f"trazas_E4_{Path(bench).stem}.jsonl"
    if not p.exists():
        continue
    for linea in p.read_text(encoding="utf-8").splitlines():
        t = json.loads(linea); comp["inputs"] += 1; n3 = 0
        for x in t["turnos"]:
            if x["tipo"] != "agente":
                continue
            if x["agente"] == "A1_Triage" and x.get("accion") == "rechazar": comp["A1_rechazos"] += 1
            if x["agente"] == "A2_Extractor" and x.get("reintento"): comp["A2_reintentos"] += 1
            if x["agente"] == "A3_Temporal":
                if x.get("accion") in ("llamar", "terminar", "invalida"): n3 += 1
                if x.get("respaldo"): comp["A3_respaldo"] += 1
            if x["agente"] == "A4_Verificador":
                comp["A4_" + x["accion"]] += 1
                comp["A4_correcciones_rechazadas"] += len(x.get("correcciones_rechazadas") or [])
        pasos.append(n3)
comp["A3_pasos_promedio"] = round(sum(pasos) / max(len(pasos), 1), 2)
print("\nComportamiento de los agentes en E4:", comp)

# frases de C y de B que E4 todavia falla
for bench in ("benchmark_independiente.json", "benchmark_heldout.json"):
    e = evaluados("E4", bench)
    if e:
        print(f"\nE4 falla en {bench}:")
        for x in e:
            if not x["es_correcto_total"]:
                print(f'  #{x["id"]:<3d} {x["texto"][:75]} -> {[(d["campo"], d["esperado"], d["obtenido"]) for d in x["detalles"] if not d["correcto"]]}')

(SALIDA / "comparacion_d3_v4.json").write_text(
    json.dumps({"tabla": tabla, "agentes_E4": comp}, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nguardado: comparacion_d3_v4.json")

## 11) Hardware + descarga

Baja `d3_resultados_intento3.zip`: resultados por brazo, **trazas de todos los inputs**, `comparacion_d3_v4.json`,
`cache_llm.json` y `hardware_intento3.json`. No repite lo que vino en `d3_previos.zip`.

In [ ]:
# 11) Hardware + zip + download
import os, zipfile, platform
from google.colab import files

props = torch.cuda.get_device_properties(0)
(SALIDA / "hardware_intento3.json").write_text(json.dumps({
    "gpu": torch.cuda.get_device_name(0), "vram_gb": round(props.total_memory / 1e9, 1),
    "vram_max_E0_modelo_base_gb": vram_e0, "vram_max_agentes_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "modelo": MODELO_ID, "adapter": ADAPTER, "dtype": "float16", "plataforma": platform.platform(),
    "torch": torch.__version__, "transformers": __import__("transformers").__version__, "peft": __import__("peft").__version__,
    "llamadas_modelo_E4_y_ablaciones": motor.llamadas, "aciertos_cache": motor.aciertos,
}, indent=2), encoding="utf-8")

RUTA_ZIP = "/content/d3_resultados_intento3.zip"
with zipfile.ZipFile(RUTA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SALIDA.rglob("*")):
        rel = p.relative_to(D3).as_posix()
        if p.is_file() and rel not in PREVIOS:
            z.write(p, rel)
print("zip con", len(zipfile.ZipFile(RUTA_ZIP).namelist()), "archivos")
files.download(RUTA_ZIP)